# ReFlex: spatial gRPC extraction (protocol v2)

Three 2D views show transmitted depth, the actual separable SOS-filter result, and geometric candidates over original depth. Missing samples are masked. ReFlex subsequently classifies absolute depth against its configured ZeroPlane, applies its MinDistance dead zone, normalises and tracks interactions.

Run in `observe` mode for live ReFlex data. Choose `demo` only while ReFlex processing is stopped: it sends two pushes and one pull through the real gRPC endpoint. Backend and framework must both support protocol v2.


In [ ]:
import asyncio
import json
import os
from pathlib import Path
import time

import grpc
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from generated_services import interactionService_pb2_grpc as rpc
from generated_services import interactionTypes_pb2 as messages

MODE = "observe"  # "demo" submits synthetic frames and changes filter settings.
DURATION_SECONDS = 60
REFRESH_SECONDS = 0.5
TARGET = os.getenv("GRPC_TARGET", "backend:50051")
SNAPSHOT = Path(os.getenv("SNAPSHOT_PATH", "/data/latest.json"))
assert MODE in {"observe", "demo"}
OPTIONS = [("grpc.max_receive_message_length", 8 * 1024 * 1024),
           ("grpc.max_send_message_length", 8 * 1024 * 1024)]

def call_rpc(method, request):
    with grpc.insecure_channel(TARGET, options=OPTIONS) as channel:
        return getattr(rpc.InteractionProcessingStub(channel), method)(request, timeout=3)

state = call_rpc("GetState", messages.StateRequest())
if state.protocol_version != 2:
    raise RuntimeError("Update backend and ReFlex together: protocol v2 required.")
print(f"Ready: {state.isReady}; frame: {state.frame_id}; mode: {MODE}")
if MODE == "demo":
    assert call_rpc("Configure", messages.ConfigRequest(cutoff=2, factor=100)).success

def send_demo_frame(elapsed):
    y, x = np.mgrid[:120, :160]
    shift = 3 * np.sin(elapsed)
    z = (1.45 - .12*np.exp(-((x-40-shift)**2+(y-35)**2)/180)
              - .10*np.exp(-((x-115)**2+(y-45-shift)**2)/180)
              + .11*np.exp(-((x-80)**2+(y-88)**2)/180))
    z[:4,:4] = np.nan
    return call_rpc("ComputeInteractions", messages.DepthValues1d(
        width=160, height=120, sample_step=1, zero_plane=1.45, z=z.ravel()))


## Live spatial view

Run for 60 seconds, rerun to continue, or use **Kernel → Interrupt Kernel**. The plots use transmitted raster coordinates; multiply X/Y by `sample_step` for sensor coordinates. Frame age indicates whether the producer has stopped.

Candidate marker shape describes geometry: **v minimum**, **^ maximum**. Colour describes the depth comparison with ZeroPlane: red Push, blue Pull, gray equal. This comparison is illustrative: the notebook does not know ReFlex's configured dead zone and therefore does not display its final accepted interactions. The wire `type` is deliberately `NONE`.


In [ ]:
started = time.monotonic()
try:
    while time.monotonic() - started < DURATION_SECONDS:
        try:
            if MODE == "demo":
                send_demo_frame(time.monotonic() - started)
            snapshot = json.loads(SNAPSHOT.read_text(encoding="utf-8"))
            width, height = snapshot["width"], snapshot["height"]
            raw = np.asarray(snapshot["raw"], dtype=float).reshape(height, width)
            filtered = np.asarray(snapshot["filtered"], dtype=float).reshape(height, width)
            plane = snapshot["zero_plane"]
            age = max(0.0, time.time() - snapshot["received_at"])
            finite = raw[np.isfinite(raw)]
            amplitude = max(float(np.max(np.abs(finite-plane))) if finite.size else 0, .001)
            fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
            titles = ["Input depth (missing samples masked)", "Separable Butterworth SOS", "Geometric candidates / depth sign"]
            for ax, values, title in zip(axes, [raw, filtered, raw], titles):
                # Preserve the original missing-data mask even after interpolation.
                masked = np.ma.array(values, mask=~np.isfinite(raw) | ~np.isfinite(values))
                im = ax.imshow(masked, origin="upper", cmap="coolwarm", vmin=plane-amplitude, vmax=plane+amplitude)
                ax.set_title(title)
                ax.set_xlabel("Raster X")
                ax.set_ylabel("Raster Y")
            for kind, marker in [("MINIMUM", "v"), ("MAXIMUM", "^")]:
                hits = [i for i in snapshot["interactions"] if i["extremum_kind"] == kind]
                colors = ["crimson" if i["z"] < plane else "royalblue" if i["z"] > plane else "gray" for i in hits]
                axes[2].scatter([i["point_idx"] % width for i in hits],
                                [i["point_idx"] // width for i in hits], c=colors,
                                marker=marker, edgecolors="white", s=90, label=f"{kind}: {len(hits)}")
            axes[2].legend(loc="upper right")
            fig.colorbar(im, ax=axes, label="Absolute sensor depth", shrink=.7)
            status = "STALE / producer stopped?" if age > 3 else "receiving"
            fig.suptitle(f"Frame {snapshot['frame_id']} | age {age:.1f}s | {status} | step {snapshot['sample_step']}\n"
                         f"ZeroPlane={plane:.4f}; filter={snapshot['config']}; depth colours exclude ReFlex dead zone")
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
            print(json.dumps(snapshot["interactions"][:10], indent=2))
        except FileNotFoundError:
            clear_output(wait=True)
            print("Waiting for a protocol-v2 frame. Start ReFlex processing or choose demo mode.")
        except KeyError as error:
            raise RuntimeError("Snapshot uses the old schema. Rebuild/restart the backend and update ReFlex.") from error
        except grpc.RpcError as error:
            clear_output(wait=True)
            print(f"gRPC {error.code().name}: {error.details()}; retrying…")
        await asyncio.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Live view stopped.")
finally:
    plt.close("all")


## Interpretation and lifecycle

The processor fills missing samples with their nearest valid neighbour for filtering only, applies an order-5 SOS filter along both axes, detects spatial extrema and verifies contrast against original neighbouring depths. Returned depths remain original measurements. A saddle between deformations is not a 2D extremum. Additional supported extrema may exist; filter ringing alone is rejected.

`sample_step = NumSkipValues + 1` applies to both axes. Effective filter sampling rate is `factor / sample_step`; require `cutoff < factor / (2 * sample_step)`. Defaults are cutoff 2 and factor 100. Saved ReFlex settings are not migrated: set these values explicitly when upgrading. The service now applies configured values when connecting.

Snapshots are atomic, contain JSON `null` for missing values, and retain one frame. Snapshot writes add processing overhead. Saved notebooks persist in the named volume, including across image rebuilds. Export edits before replacing an old notebook with this protocol-v2 version. IDs are per frame; timestamps retain the backend's Unix-millisecond convention.
